In [ ]:
# Copyright (c) 2026 Nokia Bell Labs
# Licensed under the BSD 3 Clause license
# SPDX-License-Identifier: BSD-3-Clause

In [ ]:
import re
import ast
import os, sys
import matplotlib.pyplot as plt
import numpy as np
import csv
import pandas as pd
from collections import defaultdict
from sklearn.metrics import adjusted_rand_score
import numbers

In [ ]:
def post_process_result_for_CLoVE_central_local(rounds_data, algo='CLoVE'):
    #print('In Local')
    final_result = {}
    final_result['clusters'] = {}
    final_result['accuracies'] = []
    for r, data in rounds_data.items():
        final_result['clusters'][r] = [data[0]]

    mean_accuracies = []

    for round_num, (mapping, accuracies_1, accuracies_2) in rounds_data.items():
        acc_list_2 = [accuracies_2[client][mapping[client]] for client in mapping if
                      client in accuracies_2 and mapping[client] in accuracies_2[client]]
        #print(acc_list_2)

        if acc_list_2:
            mean_accuracies.append(float(np.mean(acc_list_2)))

    final_result['accuracies'] = mean_accuracies
    return final_result

In [ ]:
def post_process_result_for_CLoVE(rounds_data, algo='CLoVE',  stable_round=None):
    if algo == 'local' or algo == 'vanillaFL' or algo == 'centralized':
        return post_process_result_for_CLoVE_central_local(rounds_data, algo)

    final_result = {}
    final_result['clusters'] = {}
    final_result['accuracies'] = []
    for r, data in rounds_data.items():
        final_result['clusters'][r] = [data[0]]
    if stable_round is not None:
        final_result['stable_round'] = stable_round

    mean_accuracies = []

    for round_num, (mapping, accuracies_1, accuracies_2) in rounds_data.items():
        acc_list_1 = [accuracies_1[client][mapping[client]] for client in mapping if
                      client in accuracies_1 and mapping[client] in accuracies_1[client]]
        acc_list_2 = [accuracies_2[client][mapping[client]] for client in mapping if
                      client in accuracies_2 and mapping[client] in accuracies_2[client]]

        if acc_list_1 :
            mean_accuracies.append(float(np.mean(acc_list_1)))

    final_result['accuracies'] = mean_accuracies
    return final_result

In [ ]:
def get_rounds_data_for_CLoVE(seed_lines, algo):
    rounds_data = {}
    aggregation_pattern = re.compile(r"Server: Aggregating model (\d+) based on local models of clients: \[(.*?)\]")
    accuracy_pattern = re.compile(r"For client (\d+) and model (\d+), test accuracy = ([0-9\.]+)")
    round_pattern = re.compile(r"Beginning of round (\d+):")
    stability_pattern = re.compile(r"Cluster stability has been achieved.")

    current_round = None
    current_mapping = {}
    current_accuracies_1 = {}
    current_accuracies_2 = {}
    recording_second_accuracy = True  # Start by looking for train accuracy lines
    found_stability = False
    stable_round = None

    for line in seed_lines:
        round_match = round_pattern.search(line)
        aggregation_match = aggregation_pattern.search(line)
        accuracy_match = accuracy_pattern.search(line)
        stability_match = stability_pattern.search(line)

        # Ignore first test set
        if round_match:
            if current_round is not None:
                rounds_data[current_round] = (current_mapping, current_accuracies_1, current_accuracies_2)

            current_round = int(round_match.group(1))
            current_mapping = {}
            current_accuracies_1 = {}  # Test accuracy
            current_accuracies_2 = {}  # Train accuracy
            recording_second_accuracy = True  # Following lines are for training accuracy
        elif aggregation_match:
            model_id = int(aggregation_match.group(1))
            clients = map(int, aggregation_match.group(2).split(', '))
            for client in clients:
                current_mapping[client] = model_id
            recording_second_accuracy = False  # Next we should get lines for test accuracy
        elif accuracy_match:
            client_id = int(accuracy_match.group(1))
            model_id = int(accuracy_match.group(2))
            accuracy = float(accuracy_match.group(3))
            if recording_second_accuracy:
                if client_id not in current_accuracies_2:
                    current_accuracies_2[client_id] = {}
                current_accuracies_2[client_id][model_id] = accuracy
            else:
                if client_id not in current_accuracies_1:
                    current_accuracies_1[client_id] = {}
                current_accuracies_1[client_id][model_id] = accuracy
        elif not found_stability and stability_match:
            # Finds first stable round
            stable_round = current_round
            found_stability = True
        elif line.strip() == "":  # Empty line marks transition to second accuracy set
            transition = True  # not used
            #recording_second_accuracy = False # Next we should get lines for test accuracy

    if current_round is not None:
        rounds_data[current_round] = (current_mapping, current_accuracies_1, current_accuracies_2)
        #print(f"round: {current_round}, clustering: {current_mapping}")

    for _, round_data in rounds_data.items():
        if round_data[0] is None or len(round_data[0]) <= 0:
            if algo == 'local':
                for i in range(len(round_data[2])):
                    round_data[0][i] = i
            else:  # this is for vanilla and centralized
                for i in range(len(round_data[2])):
                    round_data[0][i] = 0

    return rounds_data, stable_round

In [ ]:
def parse_log_for_CLoVE(file_path):
    line_data = []
    with open(file_path, 'r') as file:
        for line in file:
            line_data.append(line)
    return get_rounds_data_for_CLoVE(line_data)

In [ ]:
def process_seed_lines_for_CLoVE(seed_lines, algo='CLoVE'):
    rounds_data, stable_round = get_rounds_data_for_CLoVE(seed_lines, algo)
    # for round, data in rounds_data.items():
    #     print(round, data)
    return post_process_result_for_CLoVE(rounds_data, algo, stable_round=stable_round)

In [ ]:
def process_seed_lines_for_PACFL(seed_lines, algo='PACFL'):
    data = {
        'clusters': {},
        'accuracies': []
    }

    current_round = None
    for line in seed_lines:
        # Match round line
        round_match = re.match(r"#+ ROUND (\d+) #+", line)
        if round_match:
            current_round = int(round_match.group(1))
            # Ensure accuracy list has enough entries
            while len(data['accuracies']) < current_round:
                data['accuracies'].append(None)
            continue

        # Match clusters line
        if line.startswith("Clusters:"):
            cluster_list_str = line[len("Clusters:"):].strip()
            try:
                cluster_groups = ast.literal_eval(cluster_list_str)
                cluster_dict = {}
                for cluster_id, clients in enumerate(cluster_groups):
                    for client_id in clients:
                        cluster_dict[client_id] = cluster_id
                data['clusters'] = cluster_dict
            except Exception as e:
                print(f"Failed to parse clusters on line: {line}")
            continue

        # Match accuracy line
        acc_match = re.match(r"End round avg\. accuracy: ([\d\.]+)", line)
        if acc_match and current_round is not None:
            accuracy = float(acc_match.group(1)) / 100.0
            data['accuracies'][current_round - 1] = accuracy

    # Assign clusters to all rounds (since PACFL finds one clustering only, in the first round itself)
    clusters = {}
    for i in range(len(data['accuracies'])):
        clusters[i + 1] = [data['clusters']]
    data['clusters'] = clusters

    return data

# Example usage:
# result = parse_log_file("your_log_file.txt")
# print(result)

In [ ]:
def extract_cluster_groups_for_CFL(cluster_data_str):
    """
    Extracts a list of lists from a string like:
    '[array([1, 2]), array([3, 4])]' into [[1, 2], [3, 4]]
    """
    array_matches = re.findall(r'array\((\[[^\]]*\])\)', cluster_data_str)
    cluster_groups = []
    for arr in array_matches:
        try:
            clients = ast.literal_eval(arr)
            cluster_groups.append(clients)
        except Exception as e:
            print(f"Failed to parse array: {arr} -> {e}")
    return cluster_groups

In [ ]:
def process_seed_lines_for_CFL(seed_lines, algo='CFL'):
    data = {
        'clusters': {},  # round_num: {client_id: cluster_id}
        'accuracies': []  # index = round_num - 1
    }

    for line in seed_lines:
        line = line.strip()

        # if line.startswith("Starting run with seed="):
        #     break

        # Match clustering line
        cluster_match = re.match(r"For algo=CFL[:,]*\s*clusters[:=]*\s*(\d+),\s*(.*)", line)
        if cluster_match:
            round_num = int(cluster_match.group(1))
            cluster_data_str = cluster_match.group(2)
            try:
                cluster_groups = extract_cluster_groups_for_CFL(cluster_data_str)
                cluster_dict = {}
                for cluster_id, clients in enumerate(cluster_groups):
                    for client_id in clients:
                        cluster_dict[int(client_id)] = cluster_id
                data['clusters'][round_num] = [cluster_dict]  # ✅ Store by round
            except Exception as e:
                print(f"Cluster parse failed for round {round_num}: {e}")
            continue

        # Match accuracy line
        acc_match = re.match(r"For algo=CFL, acc=[:=]? (\d+),\s+(.*)", line)
        if acc_match:
            round_num = int(acc_match.group(1))
            acc_list_str = acc_match.group(2)
            try:
                acc_list = ast.literal_eval(acc_list_str)
                acc_list = [float(a) for a in acc_list]
                avg_acc = float(np.mean(acc_list))
                #print(acc_list, avg_acc)
                while len(data['accuracies']) < round_num:
                    data['accuracies'].append(None)
                data['accuracies'][round_num - 1] = avg_acc
            except Exception as e:
                print(f"Accuracy parse failed for round {round_num}: {e}")
            continue

    return data

In [ ]:
def process_seed_lines_fedgroup(seed_lines, algo='FedGroup'):
    data = {
        'clusters': {},  # {round_num: {client_id: cluster_id}}
        'accuracies': []  # list indexed by round number
    }

    for line in seed_lines:
        line = line.strip()

        # Match the cluster mapping line
        cluster_match = re.match(r"FedGroup: For round (\d+), all_c: (.+)", line)
        if cluster_match:
            round_num = int(cluster_match.group(1))
            cluster_data_str = cluster_match.group(2)
            try:
                cluster_list = ast.literal_eval(cluster_data_str)
                cluster_dict = {}
                for cluster_id, client_list in cluster_list:
                    for client_id in client_list:
                        cluster_dict[int(client_id)] = cluster_id
                data['clusters'][round_num] = [cluster_dict]

                ## HACK to take car of cases where initial clusters are not complete
                if round_num == 3:
                    for i in range(3):
                        data['clusters'][i] = [cluster_dict]

            except Exception as e:
                print(f"Failed to parse cluster data in round {round_num}: {e}")
            continue

        # Match the accuracy line (Partial or Complete)
        acc_match = re.match(r"FedGroup: Round (\d+), Test\((Partial|Complete)\) ACC: ([\d.]+)", line)
        if acc_match:
            round_num = int(acc_match.group(1))
            acc = float(acc_match.group(3))

            while len(data['accuracies']) <= round_num:
                data['accuracies'].append(None)
            data['accuracies'][round_num] = acc
            continue

    return data

In [ ]:
def process_seed_lines_FedPAC(seed_lines, algo='FedPAC'):
    data = {
        'clusters': {},  # {round_num: {client_id: cluster_id}}
        'accuracies': []  # list indexed by round number
    }

    for line in seed_lines:
        line = line.strip()

        # Match the cluster mapping line
        cluster_match = re.match(fr"{algo}: For round (\d+), all_c: (.+)", line)
        if cluster_match:
            round_num = int(cluster_match.group(1))
            cluster_data_str = cluster_match.group(2)
            try:
                cluster_list = ast.literal_eval(cluster_data_str)
                cluster_dict = {}
                for cluster_id, client_list in cluster_list:
                    for client_id in client_list:
                        cluster_dict[int(client_id)] = cluster_id
                data['clusters'][round_num] = [cluster_dict]
            except Exception as e:
                print(f"Failed to parse cluster data in round {round_num}: {e}")
            continue

        # Match the accuracy line (Partial or Complete)
        round_match = re.search(r'Round number: (\d+)', line)
        acc_match = re.search(r'Averaged Test Accuracy: ([0-9.]+)', line)

        if round_match:
            round_num = int(round_match.group(1))
        elif acc_match and round_num is not None:
            acc = float(acc_match.group(1))
            while len(data['accuracies']) <= round_num:
                data['accuracies'].append(None)
            data['accuracies'][round_num] = acc

    return data

In [ ]:
def process_seed_lines_pFedMe(seed_lines, algo='pFedMe'):
    data = {
        'clusters': {},  # {round_num: {client_id: cluster_id}}
        'accuracies': []  # list indexed by round number
    }

    for line in seed_lines:
        line = line.strip()

        # Match the cluster mapping line
        cluster_match = re.match(fr"{algo}: For round (\d+), all_c: (.+)", line)
        if cluster_match:
            round_num = int(cluster_match.group(1))
            cluster_data_str = cluster_match.group(2)
            try:
                cluster_list = ast.literal_eval(cluster_data_str)
                cluster_dict = {}
                for cluster_id, client_list in cluster_list:
                    for client_id in client_list:
                        cluster_dict[int(client_id)] = cluster_id
                data['clusters'][round_num] = [cluster_dict]
            except Exception as e:
                print(f"Failed to parse cluster data in round {round_num}: {e}")
            continue

        # Match the accuracy line (Partial or Complete)
        round_match = re.search(r'Round number: (\d+)', line)
        acc_match = re.search(r'Average Test Accuracy: ([0-9.]+)', line)

        if round_match:
            round_num = int(round_match.group(1))
        elif acc_match and round_num is not None:
            acc = float(acc_match.group(1))
            while len(data['accuracies']) <= round_num:
                data['accuracies'].append(None)
            data['accuracies'][round_num] = acc

    return data

In [ ]:
seed_lines = [
    "For algo=CFL:, clusters=: 1,  [array([7, 9, 10]), array([0, 1, 2])]",
    "For algo=CFL, acc=: 1, ['0.995', '0.990', '0.992', '0.981', '0.993', '0.978', '0.984', '0.969', '0.970', '0.972', '0.985', '0.986', '0.972', '0.988', '0.979']",

    "For algo=CFL:, clusters=: 2,  [array([7, 9, 10]), array([0, 1, 2])]",
    "For algo=CFL, acc=: 1, ['0.995', '0.990', '0.992', '0.981', '0.993', '0.978', '0.984', '0.969', '0.970', '0.972', '0.985', '0.986', '0.972', '0.988', '0.979']",
]
a = process_seed_lines_for_CFL(seed_lines)
print(a)

In [ ]:
def parse_log_file_with_seed_info(file_path, algo='PACFL', doing_one_algo=False):
    seed_data = {}
    current_seed = None
    seed_lines = []
    seeds_found = False

    process_seed_lines = None
    if algo == 'PACFL':
        process_seed_lines = process_seed_lines_for_PACFL
    elif algo == 'CFL':
        process_seed_lines = process_seed_lines_for_CFL
    elif algo in ['CLoVE', 'IFCA', 'local', 'vanillaFL', 'centralized']:
        process_seed_lines = process_seed_lines_for_CLoVE
    elif algo in ['FlexCFL', 'FeSEM']:
        process_seed_lines = process_seed_lines_fedgroup
    elif algo in ['FedPAC', 'PerAvg', 'FedProto', 'FedALA']:
        process_seed_lines = process_seed_lines_FedPAC
    elif algo == 'pFedMe':
        process_seed_lines = process_seed_lines_pFedMe
    else:
        print(f"Algorithm not supported: {algo}")
        return None

    skip_lines = False  # Flag to skip lines from other algorithms

    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()

            # Detect any "Starting run" line
            try:
                if not doing_one_algo:
                    any_seed_match = re.match(r"Starting run with seed=(\d+),algorithm=([^\s]+)", line)
                    if any_seed_match:
                        found_seed = int(any_seed_match.group(1))
                        found_algo = any_seed_match.group(2)
                else:
                    any_seed_match = re.match(r"Starting run with seed=(\d+)", line)
                    if any_seed_match:
                        found_seed = int(any_seed_match.group(1))
                        found_algo = algo    
            except Exception as e:
                print(f"Failed on line {line}")
                sys.exit(0)
             
            # any_seed_match = re.match(r"Starting run with seed=(\d+),algorithm=([^\s]+)", line)
            if any_seed_match:
            #     found_seed = int(any_seed_match.group(1))
            #     found_algo = any_seed_match.group(2)

                if found_algo != algo:
                    # Mismatch: enter skip mode
                    skip_lines = True
                    continue
                else:
                    # Match: end skip mode
                    skip_lines = False
                    seeds_found = True

                    # Process previous seed block
                    if current_seed is not None and seed_lines:
                        seed_data[current_seed] = process_seed_lines(seed_lines, algo)

                    # Start new seed block
                    current_seed = found_seed
                    seed_lines = []
                    continue

            # Accumulate lines only if not skipping
            if not skip_lines:
                seed_lines.append(line)

        # Final block
        if seeds_found:
            if current_seed is not None and seed_lines:
                seed_data[current_seed] = process_seed_lines(seed_lines, algo)
        else:
            # No seeds found — treat whole file as seed 1
            seed_data[1] = process_seed_lines(seed_lines)

    return seed_data

In [ ]:
def parse_log_file_with_seed_info_prev(file_path, algo='PACFL'):
    seed_data = {}
    current_seed = None
    seed_lines = []
    seeds_found = False

    process_seed_lines = None
    if algo == 'PACFL':
        process_seed_lines = process_seed_lines_for_PACFL
    elif algo == 'CFL':
        process_seed_lines = process_seed_lines_for_CFL
    elif algo == 'CLoVE' or algo == 'IFCA':
        process_seed_lines = process_seed_lines_for_CLoVE
    else:
        print(f"Algo not supported: {algo}")
        return None

    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()

            # Detect start of a new seed block
            seed_match = re.match(rf"Starting run with seed=(\d+),algorithm={algo}", line)
            if seed_match:
                seeds_found = True
                # Process previous seed block
                if current_seed is not None:
                    seed_data[current_seed] = process_seed_lines(seed_lines)

                # Start new seed block
                current_seed = int(seed_match.group(1))
                seed_lines = []
                continue

            # Accumulate lines for the current seed
            seed_lines.append(line)

        # Process the last seed block or whole file if no seeds found
        if seeds_found:
            if current_seed is not None:
                seed_data[current_seed] = process_seed_lines(seed_lines)
        else:
            # Assume all data belongs to seed 1
            seed_data[1] = process_seed_lines(seed_lines)

    return seed_data

In [ ]:
def compute_and_plot_stats(rounds_data):
    round_numbers = []
    mean_accuracies_1 = []
    mean_accuracies_2 = []

    for round_num, (mapping, accuracies_1, accuracies_2) in rounds_data.items():
        acc_list_1 = [accuracies_1[client][mapping[client]] for client in mapping if
                      client in accuracies_1 and mapping[client] in accuracies_1[client]]
        acc_list_2 = [accuracies_2[client][mapping[client]] for client in mapping if
                      client in accuracies_2 and mapping[client] in accuracies_2[client]]

        if acc_list_1 and acc_list_2:
            round_numbers.append(round_num)
            mean_accuracies_1.append(np.mean(acc_list_1))
            mean_accuracies_2.append(np.mean(acc_list_2))
            #print(f"round ={round_num}: ")
            #print(f"acc1 ={acc_list_1}: ")
            #print(f"acc2 ={acc_list_2}: ")

    plt.figure(figsize=(10, 5))
    plt.plot(round_numbers, mean_accuracies_1, label='Test accuracy', marker='o')
    plt.plot(round_numbers, mean_accuracies_2, label='Train accuracy', marker='s')
    plt.xlabel("Round Number")
    plt.ylabel("Mean Test Accuracy")
    plt.title("Mean Test Accuracy Over Rounds")
    plt.legend()
    plt.grid(True)
    plt.show()
    x = [round(float(num), 3) for num in mean_accuracies_1]
    print(f"test loss: {x}")

In [ ]:
def find_stabilization_round(rounds_data):
    final_mapping = list(rounds_data.values())[-1][0]  # Get the final client-to-model mapping
    stabilization_rounds = []

    for round_num, (mapping, _, _) in rounds_data.items():
        if mapping == final_mapping:
            stabilization_rounds.append(round_num)

    # plt.figure(figsize=(10, 5))
    # plt.hist(stabilization_rounds, bins=10, alpha=0.75, color='b', edgecolor='black')
    # plt.xlabel("Round Number")
    # plt.ylabel("Frequency of Stabilization")
    # plt.title("Rounds Where Client-to-Model Mapping Stabilizes")
    # plt.grid(True)
    # plt.show()

    return stabilization_rounds

In [ ]:
def plot_stabilization(round_data):
    """
    Plots whether the client-to-model mapping has stabilized at each round.

    :param round_data: Dictionary of rounds mapping to (client_mapping, test_acc_1, test_acc_2).
    """
    final_round = max(round_data.keys())  # Get the last round number
    final_mapping = round_data[final_round][0]  # Extract the final mapping

    rounds = sorted(round_data.keys())  # Ensure rounds are in order
    stabilization_status = [1 if round_data[r][0] == final_mapping else 0 for r in rounds]  # 1 if stabilized, else 0

    # Plot stabilization
    plt.figure(figsize=(8, 5))
    plt.plot(rounds, stabilization_status, marker='o', linestyle='-', color='b', label="Stabilization Status")

    plt.xlabel("Round Number")
    plt.ylabel("Stabilized (1) / Not Stabilized (0)")
    plt.title("Client-to-Model Mapping Stabilization Over Rounds")
    plt.yticks([0, 1], labels=["Not Stabilized", "Stabilized"])
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    plt.show()

    return stabilization_status

In [ ]:
#ADDED_NEW
def project_ground_truth(ground_truth, dict_labels):
    gt_prime = [ground_truth[i] for i in range(len(ground_truth)) if i in dict_labels]
    labels_list = [dict_labels[k] for k in sorted(dict_labels.keys())]
    return gt_prime, labels_list

In [ ]:
def compute_ari(list_labels, dict_labels):
    #ADDED_NEW
    if len(list_labels) != len(dict_labels):
        #return None
        list_labels,  dict_label_list = project_ground_truth(list_labels, dict_labels)
    else:
        # Convert dict to list using index order
        dict_label_list = [dict_labels[i] for i in range(len(list_labels))]

    # Compute ARI
    ari_score = adjusted_rand_score(list_labels, dict_label_list)
    return ari_score

In [ ]:
def plot_ARI(round_data, ground_truth, seed, algo):
    """
    Plots ARI at each round.

    :param round_data: Dictionary of rounds mapping to (client_mapping, test_acc_1, test_acc_2).
    """

    rounds = sorted(round_data.keys())  # Ensure rounds are in order
    ARI_list = [compute_ari(ground_truth, round_data[r][0]) for r in rounds]  # 1 if stabilized, else 0

    # Plot stabilization
    plt.figure(figsize=(8, 5))
    plt.plot(rounds, ARI_list, marker='o', linestyle='-', color='b', label="Adj. Rand Index")

    plt.xlabel("Round Number")
    #plt.ylabel("Stabilized (1) / Not Stabilized (0)")
    plt.title(f"Clustering Accuracy: seed={seed}, algo={algo}")
    plt.ylim(-0.5, 1.05)
    #plt.yticks([0, 1], labels=["Not Stabilized", "Stabilized"])
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    plt.show()

    return ARI_list

In [ ]:
def compute_mapping_accuracy(round_data):
    """
    Computes the accuracy of the final client-to-model mapping based on adjacency and size consistency.
    Prints accuracy and all valid sets (model: [clients]) in a single line.

    :param round_data: Dictionary of rounds mapping to (client_mapping, test_acc_1, test_acc_2).
    :return: Accuracy score (float).
    """
    final_round = max(round_data.keys())  # Get the last round number
    final_mapping = round_data[final_round][0]  # Extract the final client-to-model mapping

    # Group clients by their assigned model
    model_to_clients = defaultdict(list)
    for client, model in final_mapping.items():
        model_to_clients[model].append(client)

    # Sort clients within each model
    for model in model_to_clients:
        model_to_clients[model].sort()

    # Step 2: Check adjacency condition
    adjacent_models = set()
    for model, clients in model_to_clients.items():
        if all(clients[i] + 1 == clients[i + 1] for i in range(len(clients) - 1)):  # Clients must be consecutive
            adjacent_models.add(model)

    # Step 3: Count models with the same number of clients
    size_to_models = defaultdict(list)  # Map client group sizes to list of models with that size
    for model in adjacent_models:
        size_to_models[len(model_to_clients[model])].append(model)

    # Step 4: Find the largest set of models with the same number of clients
    max_valid_set_size = max(map(len, size_to_models.values()), default=0)  # Default to 0 if no valid models
    largest_valid_sets = [models for models in size_to_models.values() if len(models) == max_valid_set_size]

    # Step 5: Compute accuracy
    total_models = len(model_to_clients)
    accuracy = max_valid_set_size / total_models if total_models > 0 else 0

    # Print results with client lists
    valid_sets_str = " | ".join(
        [f"{model}:{model_to_clients[model]}" for models in largest_valid_sets for model in models])
    print(f"Accuracy: {accuracy:.3f} | Valid Sets: {valid_sets_str}")

    return accuracy

In [ ]:
def is_int_or_str_int(x):
    if isinstance(x, int):
        return True
    if isinstance(x, str):
        try:
            int(x)
            return True
        except ValueError:
            return False
    return False

In [ ]:
def extract_ground_truth_list(run_id, base_log_dir):
    #pattern = r"Ground Truth Client to Cluster:\s*\[([0-9,\s]+)\]"
    pattern = re.compile(r"Ground Truth Client to Cluster:\s*(\[[^\]]+\])")

    log_dir = os.path.join(base_log_dir, str(run_id))
    file_name = str(run_id) + "_pfl_experiments.log"
    file_path = os.path.join(log_dir, file_name)

    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()
            #atch = re.search(pattern, line)
           

            match = pattern.match(line)
            if match:
                raw_list_str = match.group(1)  # This will be the string inside [...]
                cluster_assignments = ast.literal_eval(raw_list_str)  # Safely turns string into list
                #print("Parsed list:", cluster_assignments)
                # Deal with FEMNIST where ground truth data contains writer names
                if not all(is_int_or_str_int(item) for item in cluster_assignments):
                    return list(range(len(cluster_assignments)))
                converted_list = [int(cluster_assignments[i]) for i in range(len(cluster_assignments))]
                return converted_list

                

    print("Line not found or could not extract list.")
    return None

In [ ]:
def extract_ground_truth_list_old(run_id, base_log_dir):
    pattern = r"Ground Truth Client to Cluster:\s*\[([0-9,\s]+)\]"

    log_dir = os.path.join(base_log_dir, str(run_id))
    file_name = str(run_id) + "_pfl_experiments.log"
    file_path = os.path.join(log_dir, file_name)

    with open(file_path, 'r') as file:
        for line in file:
            match = re.search(pattern, line)
            if match:
                number_str = match.group(1)
                number_list = [int(num.strip()) for num in number_str.split(',')]
                return number_list

    print("Line not found or could not extract list.")
    return None

In [ ]:
def get_algo_results(run_id, base_log_dir, algo='PACFL'):
    log_dir = os.path.join(base_log_dir, str(run_id))
    file_name = str(run_id) + "_pfl_experiments.log"
    file_path = os.path.join(log_dir, file_name)

    if run_id in [1901, 1902, 1903, 1900] or (run_id >= 4000 and run_id < 7000) or (run_id >= 8100 and run_id < 8200): # For ablation and early stopping, partial participation, unknown K. These are only for one algo -- CLoVE
        only_one_algo = True
    else:
        only_one_algo = False

    seeds_rounds_info = parse_log_file_with_seed_info(file_path, algo, doing_one_algo=only_one_algo)
    return seeds_rounds_info

In [ ]:
def plot_accuracy_for_run_id(run_id, base_log_dir):
    log_dir = os.path.join(base_log_dir, run_id)
    file_name = run_id + "_pfl_experiments.log"
    file_path = os.path.join(log_dir, file_name)

    rounds_info = parse_log_for_CLoVE(file_path)
    compute_and_plot_stats(rounds_info)

In [ ]:
def plot_clustering_for_run_id(run_id, base_log_dir):
    log_dir = os.path.join(base_log_dir, run_id)
    file_name = run_id + "_pfl_experiments.log"
    file_path = os.path.join(log_dir, file_name)

    rounds_info = parse_log_for_CLoVE(file_path)

    stabilization_achieved = plot_stabilization(rounds_info)
    print("Stabilization per round:", stabilization_achieved)

In [ ]:
def get_clustering_accuracy_for_run_id(run_id, base_log_dir):
    log_dir = os.path.join(base_log_dir, run_id)
    file_name = run_id + "_pfl_experiments.log"
    file_path = os.path.join(log_dir, file_name)

    rounds_info = parse_log_for_CLoVE(file_path)

    accuracy_achieved = compute_mapping_accuracy(rounds_info)
    print(f"Clustering accuracy for {run_id} is: {accuracy_achieved}")

In [ ]:
def plot_ARI_for_run_id(run_id, base_log_dir, algo='PACFL'):
    result = get_algo_results(run_id, base_log_dir, algo=algo)
    print(result)
    ground_truth = extract_ground_truth_list(run_id, base_log_dir)
    print(f"Ground Truth: {ground_truth}")
    for seed, result_for_seed in result.items():
        ARI_list = plot_ARI(result_for_seed['clusters'], ground_truth, seed, algo)
        print(f"Clustering ARIs for {run_id}:{seed} is: {ARI_list}")

In [ ]:
# def get_index_exceeding_percent(lst, percnt):
#     if lst is None or len(lst) <= 0:
#         return 100  # Return 100 or last index if list is empty
#
#     threshold = (percnt/100.0) * lst[-1]
#
#     for idx, val in enumerate(lst):
#         if val > threshold:
#             return idx+1
#     return len(lst)  # If no value exceeds the threshold


In [ ]:
def get_index_exceeding_threshold(lst, threshold):
    if lst is None or len(lst) <= 0:
        return np.nan

    for idx, val in enumerate(lst):
        if val >= threshold:
            return idx+1
    return np.nan  # If no value exceeds the threshold

In [ ]:
# def get_val_at_index_as_prcnt(lst, indx):
#     if lst is None or len(lst) <= 0:
#         return 0  # Return 0% if list is empty
#     if indx >= len(lst) - 1:
#         return 100
#     if lst[-1] <= 0.001:
#         return 0  # Even at the end the ARI is 0
#     if lst[indx] >= lst[-1]:
#         return 100.0
#     return (lst[indx]/lst[-1])*100.0

In [ ]:
def get_val_at_index(lst, indx):
    if lst is None or len(lst) <= 0:
        return np.nan

    if indx >= len(lst) - 1:
        return lst[-1]

    return lst[indx]

In [ ]:
def update_running_average(running_avg, new_list, count):
    """
    Update or initialize the running average given the current average list,
    a new list, and the number of previous lists seen.

    If running_avg is empty (i.e., first list), just return the new list as float values.
    """
    if not running_avg:
        return [float(x) for x in new_list]

    return [
        (r_avg * count + new_val) / (count + 1)
        for r_avg, new_val in zip(running_avg, new_list)
    ]

In [ ]:
def get_averaged_results(result, ground_truth):
    final_results = {
        "ARI": [],
        "accuracies": []
    }
    count = 0
    for seed, result_for_seed in result.items():
        round_cluster_data = result_for_seed['clusters']
        rounds = sorted(round_cluster_data.keys())  # Ensure rounds are in order
        ARI_list = [compute_ari(ground_truth, round_cluster_data[r][0]) for r in rounds]  # 1 if stabilized, else 0
        acc_list = result_for_seed['accuracies']
        final_results['ARI'] = update_running_average(final_results['ARI'], ARI_list, count)
        final_results['accuracies'] = update_running_average(final_results['accuracies'], acc_list, count)
        count += 1

    return final_results

In [ ]:
def get_averaged_results_with_std(result, ground_truth):
    all_ARI_results = []
    all_accuracies_results = []
    all_ARI_indx_results = []
    all_ARI_vals_results = []
    final_results = {}
    stable_rounds = []

    for seed, result_for_seed in result.items():
        round_cluster_data = result_for_seed['clusters']
        rounds = sorted(round_cluster_data.keys())  # Ensure rounds are in order
        ARI_list = [compute_ari(ground_truth, round_cluster_data[r][0]) for r in rounds]  # 1 if stabilized, else 0
        acc_list = result_for_seed['accuracies']
        all_ARI_results.append(ARI_list)  # Append ARI values for each seed
        all_accuracies_results.append(acc_list)  # Append accuracies for each seed
        ARI_indx = get_index_exceeding_threshold(ARI_list, 0.9)
        ARI_val = get_val_at_index(ARI_list, 10)
        all_ARI_indx_results.append(ARI_indx)
        all_ARI_vals_results.append(ARI_val)
        if 'stable_round' in result_for_seed:
            stable_rounds.append(result_for_seed['stable_round'])

    num_seeds = len(result.keys())

    # Calculate standard deviations for ARI and accuracies
    if len(all_ARI_results) > 0:
        final_results['ARI_mean'] = np.mean(all_ARI_results, axis=0)
        final_results['ARI_std'] = np.std(all_ARI_results, axis=0)
        final_results['accuracies_mean'] = np.mean(all_accuracies_results, axis=0)
        final_results['accuracies_std'] = np.std(all_accuracies_results, axis=0)
        final_results['ARI_indx_mean'] = np.mean(all_ARI_indx_results, axis=0)
        final_results['ARI_indx_std'] = np.std(all_ARI_indx_results, axis=0)
        final_results['ARI_val_mean'] = np.mean(all_ARI_vals_results, axis=0)
        final_results['ARI_val_std'] = np.std(all_ARI_vals_results, axis=0)
        # Make sure we got stability round for every seed
        if num_seeds == len(stable_rounds):
            final_results['stable_round_mean'] = np.mean(stable_rounds, axis=0)
            final_results['stable_round_std'] = np.std(stable_rounds, axis=0)

    
    else:
        raise ValueError("Empty experiment: no seeds detected.")

    return final_results

In [ ]:
def get_seed_averaged_results_for_algo(run_id, base_log_dir, algo='PACFL'):
    result = get_algo_results(run_id, base_log_dir, algo=algo)
    # for seed, res in result.items():
    #     print(seed, res)
    #print(result)
    ground_truth = extract_ground_truth_list(run_id, base_log_dir)
    print(f"Ground Truth: {ground_truth}")
    ret_val = get_averaged_results(result, ground_truth)
    return ret_val

In [ ]:
def sort_tuples_by_dimension(tuples_list, dimension_indices):
    """
    Sorts a list of tuples lexicographically according to specified dimensions.

    Args:
        tuples_list: The list of tuples to sort.
        dimension_indices: A list of indices indicating the dimensions to sort by.

    Returns:
        A new list containing the sorted tuples.
    """

    def custom_sort_key(tuple_):
        return [tuple_[i] for i in dimension_indices]

    sorted_tuples = sorted(tuples_list, key=custom_sort_key)
    return sorted_tuples

In [ ]:
def order_lists(lists, order):
    """Orders a list of lists based on the first element of each sublist, following a given order.

    Args:
        lists: A list of lists, where each sublist has at least one element.
        order: A list of elements that defines the desired order.

    Returns:
        A new list of lists, ordered according to the specified order.

    Raises:
        ValueError: If any element in the order list is not found as the first element of any sublist.
    """

    ordered_lists = []
    for element in order:
        for sublist in lists:
            if sublist[0] == element:
                ordered_lists.append(sublist)
                lists.remove(sublist)
                break
    else:
        raise ValueError(f"Element '{element}' not found in the first element of any sublist.")

    return ordered_lists

# # Example usage:
# lists = [['a', 1, 2], ['c', 3, 4], ['b', 5, 6], ['a', 7, 8]]
# order = ['b', 'a', 'c']
#
# ordered_lists = order_lists(lists, order)
# print(ordered_lists)  # Output: [['b', 5, 6], ['a', 1, 2], ['a', 7, 8], ['c', 3, 4]]

In [ ]:
def parse_all_results_for_exp_selector(base_log_dir, results_file, exp_selector, min_exp_id=1000, max_exp_id=4000):
    """
    Parses a CSV file containing experiment results and extracts relevant data based on a specified experiment selector.

    Args:
        results_file (str): Path to the CSV file containing experiment results.
        exp_selector (str): The experiment selector to filter results by.

    Returns:
        avg_results_with_std: A nested dictionary with keys being the datasets and the alfgorithms and values being the results
    """
    algo_keys = {   # used for renaming algorithms
        'PerAvg': 'Per-FedAvg',
        'local': 'Local-only',
        'vanillaFL': 'FedAvg',
        'centralized': 'Centralized',
    }

    ablation_runs = [1900, 1901, 1902, 1903]
    pp_runs = [5000, 5001, 5002, 5003, 5004, 5005]
    scaling_runs = [7001, 7002, 7003, 7004, 7005, 7011, 7012, 7013, 7014, 7015, 7021, 7022, 7023, 7024, 7025]

    experiment_list_df = pd.read_csv(results_file)
    selected_rows = experiment_list_df[
        (experiment_list_df['Exp Selector'] == exp_selector) &
        (experiment_list_df['Experiment ID'] >= min_exp_id) &
        (experiment_list_df['Experiment ID'] < max_exp_id)
        ]

    run_id_dataset_algo_tuples = []
    for index, row in selected_rows.iterrows():
        run_id = row['Experiment ID']
        dataset = row['Dataset']
        #### For ablations, and special categories. these all have the same dataset, so we do this hack
        if run_id in ablation_runs or run_id in pp_runs or run_id in scaling_runs:
            dataset = f"{run_id}"
        

            
        algorithms = row['Algorithm']
        algorithms_list = [item.strip() for item in algorithms.split(',')]
        run_id_dataset_algo_tuples.extend([(run_id, dataset, algo) for algo in algorithms_list])

    run_id_dataset_algo_tuples = sort_tuples_by_dimension(
        tuples_list=run_id_dataset_algo_tuples,
        dimension_indices=[1, 2]
    )

    avg_results_with_std = {}
    for (run_id, dataset, algo) in run_id_dataset_algo_tuples:
        print(f"Parsing results from run_id {run_id} and algo {algo}")
        result = get_algo_results(run_id, base_log_dir, algo=algo)
        ground_truth = extract_ground_truth_list(run_id, base_log_dir)
        algo_key = algo_keys[algo] if algo in algo_keys.keys() else algo
        if dataset not in avg_results_with_std.keys():
            avg_results_with_std[dataset] = {}
        avg_results_with_std[dataset][algo_key] = get_averaged_results_with_std(result, ground_truth)

    return avg_results_with_std

In [ ]:
def modify_dataset_labels(datasets_list):
    ds_mapping = {'1900': "orig", '1901': 'no_matching', '1902': 'agg_clustering', '1903': 'squareroot_loss', 
                  '5000':'MNIST_1.0', '5001':'MNIST_0.9', 
                  '5002':'MNIST_0.75', '5003':'MNIST_0.5',
                  '5004':'MNIST_0.25', '5005':'MNIST_0.1',
                  '7001':'MNIST_50', '7002':'MNIST_100', '7003':'MNIST_250', '7004':'MNIST_500', '7005':'MNIST_1000',
                  '7011':'CIFAR10_50', '7012':'CIFAR10_100', '7013':'CIFAR10_250', '7014':'CIFAR10_500', '7015':'CIFAR10_1000',
                  '7021':'FMNIST_50', '7022':'FMNIST_100', '7023':'FMNIST_250', '7024':'FMNIST_500', '7025':'FMNIST_1000'}
    new_ds_list = [ds_mapping[x] if x in ds_mapping else x for x in datasets_list]
    return new_ds_list


In [ ]:
def get_label_for_metric(metric):
    if 'ARI' in metric:
        return 'ARI'
    return metric

In [ ]:
def create_results_table(output_dir, results, metric, exp_selector):
    """
    Creates a table with mean ± std values for each algorithm and dataset.

    Args:
        results: A dictionary containing results for each algorithm and dataset.
        metric: The metric of interest.
        exp_selector: The name of the experiment.

    Returns:
        None. Writes the table to a CSV file.
    """
    print(f"----------------------\nStarting experiment {exp_selector}.")
    if len(results) == 0:
        # print(f"ERROR: No results found for experiment {exp_selector}. Skipping.")
        # return
        raise ValueError(f"No results found for experiment {exp_selector}.")

    extended_datasets = ['1900', '1901', '1902', '1903', '5000', '5001', '5002', '5003', '5004', '5005']
    extended_datasets2 =  ["7001", "7002", "7003", "7004", "7005", "7011", "7012", "7013", "7014", "7015", "7021", "7022", "7023", "7024", "7025"]
    all_algorithms_in_order = ['FedAvg', 'Local-only', 'Centralized', 'Per-FedAvg', 'FedProto', 'FedALA', 'FedPAC', 'CFL', 'FeSEM', 'FlexCFL', 'PACFL', 'IFCA', 'CLoVE']
    all_datasets_in_order = ['MNIST', 'CIFAR10', 'FMNIST', 'FEMNIST', 'AmazonReview', 'AG_news']
    all_datasets_in_order.extend(extended_datasets)
    all_datasets_in_order.extend(extended_datasets2)

    output_filename = f"supervised_table_{exp_selector}_{metric}.csv"
    with (open(os.path.join(output_dir, output_filename), 'w', newline='') as csvfile):
        writer = csv.writer(csvfile)

        # all_datasets = list(set([tuple[1] for tuple in run_id_dataset_algo_tuples]))
        # all_algorithms = list(results[all_datasets[0]].keys())
        # if re.match(pattern=r"^[0-9]+$", string=list(results.keys())[0]):     # if ablation experiment
        #     all_datasets_for_table = list(results.keys())
        # else:
        #     all_datasets_for_table = [d for d in all_datasets_in_order if d in list(results.keys())]
        #     all_algorithms_for_table = [a for a in all_algorithms_in_order if a in list(results[all_datasets_for_table[0]].keys())]
            # all_algorithms = list(set([tuple[2] for tuple in run_id_dataset_algo_tuples]))
        all_datasets_for_table = [d for d in all_datasets_in_order if d in list(results.keys())]
        all_algorithms_for_table = [a for a in all_algorithms_in_order if a in list(results[all_datasets_for_table[0]].keys())]

        ds_labels = modify_dataset_labels(all_datasets_for_table)
        # Write header row
        header = ['Algorithm']
        header.extend(ds_labels)
        writer.writerow(header)

        # Write data rows
        for algo in all_algorithms_for_table:
            print(f"\nStarting algo = {algo}.")
            if algo == 'Centralized':
                continue        # skip centralized completely
            if metric in ['ARI', 'ARI_indx', 'ARI_val']  and algo in ['Per-FedAvg', 'FedProto', 'FedALA', 'FedPAC', 'Local-only', 'FedAvg']:
                continue        # skip ARI for general PFL (non-CFL) algorithms

            row = [algo]
            for dataset in all_datasets_for_table:
                print(f"Starting dataset = {dataset}.")
                #metric_label = get_label_for_metric(metric)
                means = results[dataset][algo][f'{metric}_mean']
                stds = results[dataset][algo][f'{metric}_std']
                if not isinstance(means, numbers.Number): # isinstance(means, list):
                    if len(means) > 0:
                        if metric == 'accuracies':  # multiply by 100
                            last_round_mean = means[-1] * 100
                            last_round_std = stds[-1] * 100
                            row.append(f"{last_round_mean:.1f} ± {last_round_std:.1f}")
                        elif metric == 'ARI':       # leave numbers as they are (for ARI)
                            last_round_mean = means[-1]
                            last_round_std = stds[-1]
                            row.append(f"{last_round_mean:.2f} ± {last_round_std:.2f}")
                elif metric in ['ARI_indx', 'stable_round']:
                    row.append(f"{means:.1f} ± {stds:.1f}")
                elif metric in ['ARI_val']:
                    row.append(f"{means:.2f} ± {stds:.2f}")

            writer.writerow(row)

    print(f"Successfully processed results for exp_selector={exp_selector} and stored them to file {output_filename}.\n")

In [ ]:
# all_algorithms = ['CLoVE', 'IFCA', 'local', 'vanillaFL', 'centralized', 'CFL', 'PACFL', 'FedPAC', 'FlexCFL', 'FeSEM', 'PerAvg', 'FedProto', 'FedALA']
# all_datasets = ['MNIST', 'CIFAR10', 'FMNIST', 'AmazonReview', 'AG_news']

# run_id = 1002
# avg_results = get_seed_averaged_results_for_algo(run_id, base_log_dir, algo)
# print(avg_results)

In [ ]:
directory_path = "../"
results_dir = 'results'
output_dir = os.path.join(directory_path, results_dir)
# base_log_dir = os.path.join(directory_path, results_dir)
base_log_dir = os.path.join(directory_path, results_dir)
experiment_list_csv_file = os.path.join(directory_path, "experiment_driver", "experiment_list.csv")

In [ ]:
for exp_selector in [
    "exp111",
    "exp203",
    "exp301",
    "exp201",
    "exp401",
    "expAmazon",

    # For appendix
    "exp501",
    "exp202",

]:
    avg_results_with_std = parse_all_results_for_exp_selector(base_log_dir, experiment_list_csv_file, exp_selector, min_exp_id=1000, max_exp_id=4000)
    for metric in ['ARI', 'accuracies', 'ARI_indx', 'ARI_val']:
        create_results_table(
            output_dir=output_dir,
            results=avg_results_with_std,
            metric=metric,
            exp_selector=exp_selector,
        )

In [ ]:
# Ablation studies
for exp_selector in [
    "exp602",
]:
    avg_results_with_std = parse_all_results_for_exp_selector(base_log_dir, experiment_list_csv_file, exp_selector, min_exp_id=1000, max_exp_id=4000)
    for metric in ['ARI', 'accuracies']:
        create_results_table(
            output_dir=output_dir,
            results=avg_results_with_std,
            metric=metric,
            exp_selector=exp_selector,
        )

Newer Experiments

In [ ]:
directory_path = "../"
results_dir = 'results'
out_sub_dir = 'extensions'
output_dir = os.path.join(directory_path, results_dir, out_sub_dir)
os.makedirs(output_dir, exist_ok=True)
# base_log_dir = os.path.join(directory_path, results_dir)
base_log_dir = os.path.join(directory_path, results_dir)
experiment_list_csv_file = os.path.join(directory_path, "experiment_driver", "experiment_list.csv")

In [ ]:
for exp_selector in [
    "exp701",
]:
    avg_results_with_std = parse_all_results_for_exp_selector(base_log_dir, experiment_list_csv_file, exp_selector, min_exp_id=4000, max_exp_id=7000)
    for metric in ['ARI', 'accuracies', 'ARI_indx', 'ARI_val']:
        create_results_table(
            output_dir=output_dir,
            results=avg_results_with_std,
            metric=metric,
            exp_selector=exp_selector,
        )

In [ ]:
for exp_selector in [
    "exp111",
    "exp201",
    "exp203",
    "exp301"
]:
    avg_results_with_std = parse_all_results_for_exp_selector(base_log_dir, experiment_list_csv_file, exp_selector, min_exp_id=4000, max_exp_id=7000)
    for metric in ['ARI', 'accuracies', 'ARI_indx', 'ARI_val', 'stable_round']:
        create_results_table(
            output_dir=output_dir,
            results=avg_results_with_std,
            metric=metric,
            exp_selector=exp_selector,
        )

In [ ]:
for exp_selector in [
    "expFEMNIST",
]:
    avg_results_with_std = parse_all_results_for_exp_selector(base_log_dir, experiment_list_csv_file, exp_selector, min_exp_id=4000, max_exp_id=7000)
    for metric in ['ARI', 'accuracies', 'ARI_indx', 'ARI_val']:
        create_results_table(
            output_dir=output_dir,
            results=avg_results_with_std,
            metric=metric,
            exp_selector=exp_selector,
        )

In [ ]:
for exp_selector in [
    "exp203",
]:
    avg_results_with_std = parse_all_results_for_exp_selector(base_log_dir, experiment_list_csv_file, exp_selector, min_exp_id=7000, max_exp_id=8000)
    for metric in ['ARI', 'accuracies', 'ARI_indx', 'ARI_val']:
        create_results_table(
            output_dir=output_dir,
            results=avg_results_with_std,
            metric=metric,
            exp_selector=exp_selector,
        )

For ICML 2026

In [ ]:
directory_path = "../"
results_dir = 'results/finch_icml_seed3/finch'
output_dir = os.path.join(directory_path, results_dir)
# base_log_dir = os.path.join(directory_path, results_dir)
base_log_dir = os.path.join(directory_path, results_dir)
experiment_list_csv_file = os.path.join(directory_path, "experiment_driver", "experiment_list.csv")

In [ ]:
# Ablation studies
for exp_selector in [
    "exp602",
]:
    avg_results_with_std = parse_all_results_for_exp_selector(base_log_dir, experiment_list_csv_file, exp_selector, min_exp_id=8100, max_exp_id=9000)
    for metric in ['ARI', 'accuracies','ARI_indx', 'ARI_val']:
        create_results_table(
            output_dir=output_dir,
            results=avg_results_with_std,
            metric=metric,
            exp_selector=exp_selector,
        )

In [ ]:
directory_path = "../"
results_dir = 'results/finch_icml_seed3/kmeans'
output_dir = os.path.join(directory_path, results_dir)
# base_log_dir = os.path.join(directory_path, results_dir)
base_log_dir = os.path.join(directory_path, results_dir)
experiment_list_csv_file = os.path.join(directory_path, "experiment_driver", "experiment_list.csv")

In [ ]:
# Ablation studies
for exp_selector in [
    "exp602",
]:
    avg_results_with_std = parse_all_results_for_exp_selector(base_log_dir, experiment_list_csv_file, exp_selector, min_exp_id=8100, max_exp_id=9000)
    for metric in ['ARI', 'accuracies','ARI_indx', 'ARI_val']:
        create_results_table(
            output_dir=output_dir,
            results=avg_results_with_std,
            metric=metric,
            exp_selector=exp_selector,
        )